# Operza API Benchmark & Latency Distribution Analysis
This notebook analyzes k6 load test results and latency distributions across Operza API endpoints.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Sample simulated benchmark data representing Operza k6 checkout & ingress workload
np.random.seed(42)
endpoints = ['/api/crm/contacts', '/api/payments/checkout/link', '/api/webhooks/stripe', '/api/copilot/message']
data = []
for ep in endpoints:
    base_lat = 15 if 'contacts' in ep else (25 if 'checkout' in ep else 45)
    latencies = np.random.exponential(scale=base_lat, size=500) + 5
    for lat in latencies:
        data.append({'endpoint': ep, 'latency_ms': lat})

df = pd.DataFrame(data)
summary = df.groupby('endpoint')['latency_ms'].agg([
    ('p50', lambda x: np.percentile(x, 50)),
    ('p90', lambda x: np.percentile(x, 90)),
    ('p95', lambda x: np.percentile(x, 95)),
    ('p99', lambda x: np.percentile(x, 99)),
    ('mean', 'mean')
]).round(2)

print('=== OPERZA BENCHMARK LATENCY SUMMARY (ms) ===')
print(summary)

plt.figure(figsize=(10, 5))
for ep in endpoints:
    subset = df[df['endpoint'] == ep]['latency_ms']
    plt.hist(subset, bins=30, alpha=0.6, label=ep)
plt.title('Operza API Endpoint Latency Distribution')
plt.xlabel('Latency (ms)')
plt.ylabel('Request Frequency')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()